# 05 A/B Test Design - Fulfillment SLA Experiment

本 notebook 设计一个更接近真实工作的实验：针对首单用户的履约 SLA 加速方案，验证它是否能改善用户体验并提升后续复购。

重点包含三部分：

- 用历史数据抽取实验基线
- 对 `30 天复购` 和关键过程指标做样本量估算
- 给出建议的 MDE、周期和分阶段读数方案


In [ ]:
from pathlib import Path
import math
import sqlite3
from statistics import NormalDist

ROOT = Path.cwd()
if not (ROOT / 'sql').exists():
    ROOT = ROOT.parent
DB_PATH = ROOT / 'data' / 'raw' / 'olist.sqlite'

if not DB_PATH.exists():
    raise FileNotFoundError(f'Missing database: {DB_PATH}')

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)
conn.row_factory = sqlite3.Row
print(f'Connected to: {DB_PATH}')


## 1. Baseline Metrics

实验人群定义为：`首个有效订单用户`。因为我们要观察“首单履约体验是否影响 30 天内再次下单”，所以用首单作为暴露点最合理。

这里同时计算：

- `30d repeat rate`：首单后 30 天内再次下单
- `60d repeat rate`：作为更长期参考
- `late share`：晚于承诺日期签收的占比
- `4d+ late share`：严重晚到占比
- `bad review share`：评价分 `<= 2`
- `avg daily new users`：用来估算招募周期


In [ ]:
BASELINE_SQL = '''
WITH pay AS (
  SELECT order_id, SUM(payment_value) AS paid_value
  FROM order_payments
  GROUP BY 1
),
valid_orders AS (
  SELECT
    o.order_id,
    c.customer_unique_id,
    o.order_purchase_timestamp,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    CASE
      WHEN o.order_estimated_delivery_date IS NOT NULL
       AND o.order_delivered_customer_date IS NOT NULL
      THEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date)
      ELSE NULL
    END AS delay_days
  FROM orders o
  LEFT JOIN customers c
    ON o.customer_id = c.customer_id
  LEFT JOIN pay
    ON o.order_id = pay.order_id
  WHERE o.order_status NOT IN ('canceled', 'unavailable')
    AND COALESCE(pay.paid_value, 0) > 0
    AND c.customer_unique_id IS NOT NULL
    AND o.order_purchase_timestamp IS NOT NULL
),
review_dedup AS (
  SELECT
    order_id,
    review_score,
    ROW_NUMBER() OVER (
      PARTITION BY order_id
      ORDER BY review_answer_timestamp DESC, review_creation_date DESC, review_id DESC
    ) AS rn
  FROM order_reviews
),
review_final AS (
  SELECT order_id, review_score
  FROM review_dedup
  WHERE rn = 1
),
ordered AS (
  SELECT
    vo.*,
    rf.review_score,
    LEAD(vo.order_purchase_timestamp) OVER (
      PARTITION BY vo.customer_unique_id
      ORDER BY vo.order_purchase_timestamp, vo.order_id
    ) AS next_purchase_ts,
    ROW_NUMBER() OVER (
      PARTITION BY vo.customer_unique_id
      ORDER BY vo.order_purchase_timestamp, vo.order_id
    ) AS order_rank,
    MAX(vo.order_purchase_timestamp) OVER () AS max_purchase_ts
  FROM valid_orders vo
  LEFT JOIN review_final rf
    ON vo.order_id = rf.order_id
),
eligible_first_orders AS (
  SELECT *
  FROM ordered
  WHERE order_rank = 1
    AND julianday(max_purchase_ts) - julianday(order_purchase_timestamp) >= 60
),
daily_first_orders AS (
  SELECT date(order_purchase_timestamp) AS dt, COUNT(*) AS users
  FROM (
    SELECT *
    FROM ordered
    WHERE order_rank = 1
  )
  GROUP BY 1
)
SELECT
  (SELECT COUNT(*) FROM eligible_first_orders) AS eligible_users,
  (SELECT AVG(CASE WHEN next_purchase_ts IS NOT NULL AND julianday(next_purchase_ts) <= julianday(order_purchase_timestamp) + 30 THEN 1.0 ELSE 0 END) FROM eligible_first_orders) AS repeat_30d,
  (SELECT AVG(CASE WHEN next_purchase_ts IS NOT NULL AND julianday(next_purchase_ts) <= julianday(order_purchase_timestamp) + 60 THEN 1.0 ELSE 0 END) FROM eligible_first_orders) AS repeat_60d,
  (SELECT AVG(CASE WHEN delay_days > 0 THEN 1.0 ELSE 0 END) FROM eligible_first_orders) AS late_share,
  (SELECT AVG(CASE WHEN delay_days > 4 THEN 1.0 ELSE 0 END) FROM eligible_first_orders) AS late_4d_share,
  (SELECT AVG(CASE WHEN review_score <= 2 THEN 1.0 ELSE 0 END) FROM eligible_first_orders) AS bad_review_share,
  (SELECT AVG(users) FROM daily_first_orders) AS avg_daily_new_users
'''

baseline = dict(conn.execute(BASELINE_SQL).fetchone())
for key, value in baseline.items():
    if isinstance(value, float):
        print(f'{key}: {value:.4f}')
    else:
        print(f'{key}: {value}')


## 2. Proposed Experiment

实验主题：`首单履约 SLA 加速`。

一个更接近真实工作的 treatment 可以是：

- 优先分配更快的发货波次
- 对支持多承运商的订单选择更快线路
- 对高风险晚到订单增加主动异常处理

控制组保持现有履约逻辑。实验单位建议用 `首单用户`，在支付成功后随机化，这样主指标 `30 天复购` 与暴露点对齐。


## 3. Sample Size Function

这里使用双侧双样本比例检验的常见近似公式，假设：

- `alpha = 0.05`
- `power = 0.80`
- `50 / 50` 分流

样本量输出为 `每组所需样本数`。


In [ ]:
def sample_size_two_proportions(p1, p2, alpha=0.05, power=0.80):
    z_alpha = NormalDist().inv_cdf(1 - alpha / 2)
    z_beta = NormalDist().inv_cdf(power)
    p_bar = (p1 + p2) / 2
    numerator = (
        z_alpha * math.sqrt(2 * p_bar * (1 - p_bar))
        + z_beta * math.sqrt(p1 * (1 - p1) + p2 * (1 - p2))
    ) ** 2
    return math.ceil(numerator / ((p2 - p1) ** 2))


def runtime_days(total_sample, daily_users):
    return total_sample / daily_users


repeat_30d = baseline['repeat_30d']
late_share = baseline['late_share']
bad_review_share = baseline['bad_review_share']
daily_users = baseline['avg_daily_new_users']

print('repeat_30d baseline:', round(repeat_30d, 4))
print('late_share baseline:', round(late_share, 4))
print('bad_review_share baseline:', round(bad_review_share, 4))
print('avg_daily_new_users:', round(daily_users, 1))


## 4. MDE Scenarios

对低频平台来说，`30 天复购` 很难被短周期实验快速 detect 到。因此这里同时展示：

- 慢指标：`30d repeat`
- 快指标：`late share`、`bad review share`

这也是实际工作里常见的双层读数方式：先看过程指标是否按预期改善，再继续累计业务指标。


In [ ]:
repeat_mdes = [0.003, 0.004, 0.005]
late_mdes = [0.010, 0.015, 0.020]
bad_review_mdes = [0.010, 0.015, 0.020]

print('30d repeat scenarios')
for mde in repeat_mdes:
    n_per_group = sample_size_two_proportions(repeat_30d, repeat_30d + mde)
    total_n = n_per_group * 2
    print(
        f'  MDE={mde:.3%} | target={(repeat_30d + mde):.3%} | per_group={n_per_group} | total={total_n} | recruit_days={runtime_days(total_n, daily_users):.1f}'
    )

print('\nlate share scenarios')
for mde in late_mdes:
    n_per_group = sample_size_two_proportions(late_share, late_share + mde)
    total_n = n_per_group * 2
    print(
        f'  abs_change={mde:.3%} | per_group={n_per_group} | total={total_n} | recruit_days={runtime_days(total_n, daily_users):.1f}'
    )

print('\nbad review scenarios')
for mde in bad_review_mdes:
    n_per_group = sample_size_two_proportions(bad_review_share, bad_review_share + mde)
    total_n = n_per_group * 2
    print(
        f'  abs_change={mde:.3%} | per_group={n_per_group} | total={total_n} | recruit_days={runtime_days(total_n, daily_users):.1f}'
    )


## 5. Recommended Readout Plan

如果把这个实验当成一个真实运营项目，我会这样定：

- `北极星主指标`：`30 天复购率`
- `主过程指标`：`晚到率` 或 `4 天以上晚到率`
- `关键护栏`：差评率、退款率、客服联系量、物流成本/单

由于历史 Olist 量级下 `30d repeat` 太稀疏，推荐把 `+0.50pp` 作为最小值得检测的业务提升：

- 从 `1.62%` 提升到 `2.12%`
- 约等于 `+30.8%` 的相对提升
- 每组需要约 `11.5k` 用户

这已经是一个较高门槛，但对需要真实物流投入的 SLA 项目来说更合理，因为太小的 uplift 往往覆盖不了成本。


In [ ]:
recommended_repeat_mde = 0.005
recommended_late_mde = 0.015

repeat_n = sample_size_two_proportions(repeat_30d, repeat_30d + recommended_repeat_mde)
repeat_total = repeat_n * 2
repeat_days = runtime_days(repeat_total, daily_users)

late_n = sample_size_two_proportions(late_share, late_share + recommended_late_mde)
late_total = late_n * 2
late_days = runtime_days(late_total, daily_users)

print('Recommended business MDE for 30d repeat')
print(f'  baseline={repeat_30d:.3%}')
print(f'  target={(repeat_30d + recommended_repeat_mde):.3%}')
print(f'  per_group={repeat_n}')
print(f'  total={repeat_total}')
print(f'  recruit_days_at_historical_volume={repeat_days:.1f}')
print(f'  full_readout_days_with_30d_outcome_window={repeat_days + 30:.1f}')

print('\nRecommended process-gate MDE for late share')
print(f'  baseline={late_share:.3%}')
print(f'  target={(late_share - recommended_late_mde):.3%} if we interpret it as a reduction')
print(f'  per_group={late_n}')
print(f'  total={late_total}')
print(f'  recruit_days_at_historical_volume={late_days:.1f}')


## 6. Recommendation

基于当前历史量级，我不建议只靠 `30 天复购` 单独做一次性 go / no-go。更现实的做法是：

1. 先以 `晚到率` 作为阶段性放量门槛，确认机制真的生效。
2. 同时持续保留 holdout，累计 `30 天复购` 作为最终商业结果。
3. 若未来真实线上流量显著高于 Olist 历史量级，可以继续用同一套公式重新估算周期。

这类设计比“只看一次复购结果再决定”更符合真实实验实践，因为它兼顾了业务结果、机制验证和执行成本。
